<a href="https://colab.research.google.com/github/yeye080/last-mile-route-optimizer/blob/main/optimizador_rutas_ultima_milla.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#@title OPTIMIZADOR DE RUTAS DE ÚLTIMA MILLA 1.0

!pip install folium geopy


import folium
from geopy.distance import geodesic
import ipywidgets as widgets
from IPython.display import display, clear_output
import random
import pandas as pd

# 1. COORDENADAS BASE (Centrado en Barcelona como ejemplo)
COORDS_DEPOT = (41.3851, 2.1734)  # Plaza Cataluña (Almacén Central)

# Base de datos sintética de clientes potenciales en Barcelona
clientes_disponibles = [
    {"ID": 1, "Nombre": "Cliente Eixample", "Lat": 41.3925, "Lon": 2.1610, "Demanda_kg": 45},
    {"ID": 2, "Nombre": "Cliente Gracia", "Lat": 41.4026, "Lon": 2.1572, "Demanda_kg": 20},
    {"ID": 3, "Nombre": "Cliente Poblenou", "Lat": 41.3995, "Lon": 2.2031, "Demanda_kg": 80},
    {"ID": 4, "Nombre": "Cliente Sants", "Lat": 41.3780, "Lon": 2.1385, "Demanda_kg": 65},
    {"ID": 5, "Nombre": "Cliente Les Corts", "Lat": 41.3882, "Lon": 2.1282, "Demanda_kg": 110},
    {"ID": 6, "Nombre": "Cliente Sagrada Familia", "Lat": 41.4036, "Lon": 2.1744, "Demanda_kg": 35},
    {"ID": 7, "Nombre": "Cliente Raval", "Lat": 41.3792, "Lon": 2.1685, "Demanda_kg": 50},
    {"ID": 8, "Nombre": "Cliente Barceloneta", "Lat": 41.3783, "Lon": 2.1901, "Demanda_kg": 95},
    {"ID": 9, "Nombre": "Cliente Sarria", "Lat": 41.3992, "Lon": 2.1213, "Demanda_kg": 40},
    {"ID": 10, "Nombre": "Cliente Horta", "Lat": 41.4302, "Lon": 2.1612, "Demanda_kg": 15}
]

# 2. ALGORITMO DE OPTIMIZACIÓN (Nearest Neighbor / Vecino más cercano)
def optimizar_ruta_reparto(num_entregas, consumo_100km, precio_litro):
    # Selección de clientes para esta ruta
    clientes_ruta = clientes_disponibles[:num_entregas]

    no_visitados = clientes_ruta.copy()
    ruta_optima = []
    punto_actual = {"Nombre": "Almacén Central (Depósito)", "Lat": COORDS_DEPOT[0], "Lon": COORDS_DEPOT[1]}
    distancia_total = 0.0
    carga_total = 0.0

    # Bucle del algoritmo
    while no_visitados:
        # Buscar el cliente no visitado más cercano al punto actual
        mas_cercano = None
        distancia_minima = float('inf')

        for cliente in no_visitados:
            dist = geodesic((punto_actual["Lat"], punto_actual["Lon"]), (cliente["Lat"], cliente["Lon"])).kilometers
            if dist < distancia_minima:
                distancia_minima = dist
                mas_cercano = cliente

        # "Viajar" al cliente más cercano
        distancia_total += distancia_minima
        punto_actual = mas_cercano
        carga_total += mas_cercano["Demanda_kg"]
        ruta_optima.append(mas_cercano)
        no_visitados.remove(mas_cercano)

    # Regresar al almacén central
    dist_regreso = geodesic((punto_actual["Lat"], punto_actual["Lon"]), COORDS_DEPOT).kilometers
    distancia_total += dist_regreso

    # 3. CÁLCULO DE MÉTRICAS LOGÍSTICAS e IMPACTO AMBIENTAL
    combustible_consumido = (distancia_total / 100.0) * consumo_100km
    coste_combustible = combustible_consumido * precio_litro
    # 1 litro de diésel emite aprox 2.64 kg de CO2
    emisiones_co2 = combustible_consumido * 2.64

    # --- MOSTRAR MÉTRICAS EN PANTALLA ---
    print("=" * 70)
    print("🚚 INFORME DE OPTIMIZACIÓN DE RUTA - ÚLTIMA MILLA")
    print("=" * 70)
    print(f"📍 Punto de salida:      Almacén Central (Plaza Cataluña)")
    print(f"📦 Paquetes entregados:  {len(ruta_optima)} clientes")
    print(f"⚖️  Carga total a bordo:  {carga_total:.1f} kg")
    print(f"🛣️  Distancia total:      {distancia_total:.2f} km (incluido retorno)")
    print("-" * 70)
    print(f"⛽ Consumo combustible:  {combustible_consumido:.2f} L (Est. {consumo_100km} L/100km)")
    print(f"💰 Coste aproximado:     {coste_combustible:.2f} € (a {precio_litro:.2f} €/L)")
    print(f"🌱 Huella de Carbono:    {emisiones_co2:.2f} kg CO₂")
    print("=" * 70)

    print("\n📋 SECUENCIA DE PASO RECOMENDADA:")
    print("➡️ [SALIDA] Almacén Central")
    for i, nodo in enumerate(ruta_optima, 1):
        print(f"   👉 [Paso {i}] {nodo['Nombre']} (Entregar {nodo['Demanda_kg']} kg)")
    print("➡️ [LLEGADA] Regreso al Almacén")
    print("-" * 70)

    # 4. GENERAR MAPA INTERACTIVO CON FOLIUM
    mapa = folium.Map(location=COORDS_DEPOT, zoom_start=13, control_scale=True)

    # Marcador Almacén (Icono especial de casa/fábrica)
    folium.Marker(
        location=COORDS_DEPOT,
        popup="<b>🏭 Almacén Central (Origen/Destino)</b>",
        tooltip="Depósito Central",
        icon=folium.Icon(color="red", icon="home")
    ).add_to(mapa)

    # Dibujar clientes y trazar línea de la ruta
    puntos_ruta = [COORDS_DEPOT]

    for i, cliente in enumerate(ruta_optima, 1):
        coords_cliente = (cliente["Lat"], cliente["Lon"])
        puntos_ruta.append(coords_cliente)

        # Marcadores de clientes
        folium.Marker(
            location=coords_cliente,
            popup=f"<b>👤 {cliente['Nombre']}</b><br>Entrega: {cliente['Demanda_kg']} kg<br>Orden de visita: #{i}",
            tooltip=f"Parada #{i}",
            icon=folium.Icon(color="blue", icon="shopping-cart", prefix="fa")
        ).add_to(mapa)

    puntos_ruta.append(COORDS_DEPOT) # Cerrar la ruta de vuelta al depósito

    # Dibujar la línea de la ruta en el mapa
    folium.PolyLine(
        locations=puntos_ruta,
        color="#2b5c8f",
        weight=4,
        opacity=0.8,
        tooltip="Ruta de reparto optimizada"
    ).add_to(mapa)

    # --- SEGUNDA ALTERNATIVA SEGURA (Incrustación directa de HTML) ---
    from IPython.display import HTML

    mapa_html = mapa._repr_html_()
    display(HTML(f'<div style="width:100%; height:500px;">{mapa_html}</div>'))

# 5. CONSTRUCCIÓN DE LA INTERFAZ DE USUARIO
salida_mapa = widgets.Output()

html_titulo = widgets.HTML("<h3>📍 Planificador de Rutas de Última Milla</h3>")
in_entregas = widgets.IntSlider(value=5, min=3, max=10, description='Nº Entregas:')
in_consumo = widgets.FloatSlider(value=8.5, min=5.0, max=15.0, step=0.1, description='Consumo L/100km:')
in_precio = widgets.FloatText(value=1.58, description='Precio Diésel (€/L):')

boton_optimizar = widgets.Button(description='Calcular Ruta Óptima', button_style='primary', icon='map-marked-alt')

def lanzar_optimizacion(b):
    with salida_mapa:
        clear_output(wait=True)
        optimizar_ruta_reparto(in_entregas.value, in_consumo.value, in_precio.value)

boton_optimizar.on_click(lanzar_optimizacion)

interfaz_rutas = widgets.VBox([
    html_titulo,
    widgets.HBox([in_entregas, in_consumo]),
    widgets.HBox([in_precio]),
    widgets.Label(''),
    boton_optimizar
])

print("🧭 MAPAS Y ALGORITMO DE LOGÍSTICA DE REPARTO ACTIVADOS")
display(interfaz_rutas, salida_mapa)

🧭 MAPAS Y ALGORITMO DE LOGÍSTICA DE REPARTO ACTIVADOS


Output()

In [ ]:
#@title OPTIMIZADOR DE FLOTAS Y ÚLTIMA MILLA 2.0 (CORREGIDO)
# =====================================================================

import folium
from geopy.distance import geodesic
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import pandas as pd

# 1. COORDENADAS BASE (Almacén Central - Plaza Cataluña, Barcelona)
COORDS_DEPOT = (41.3851, 2.1734)

# Clientes con peso de entrega (kg) y ventana horaria máxima (horas en formato decimal, ej: 11.5 = 11:30)
clientes_v2 = [
    {"ID": 1, "Nombre": "Eixample", "Lat": 41.3925, "Lon": 2.1610, "Demanda_kg": 60, "Limite_Hora": 10.0, "Detalle_Hora": "10:00"},
    {"ID": 2, "Nombre": "Gracia", "Lat": 41.4026, "Lon": 2.1572, "Demanda_kg": 40, "Limite_Hora": 11.0, "Detalle_Hora": "11:00"},
    {"ID": 3, "Nombre": "Poblenou", "Lat": 41.3995, "Lon": 2.2031, "Demanda_kg": 90, "Limite_Hora": 12.0, "Detalle_Hora": "12:00"},
    {"ID": 4, "Nombre": "Sants", "Lat": 41.3780, "Lon": 2.1385, "Demanda_kg": 70, "Limite_Hora": 10.5, "Detalle_Hora": "10:30"},
    {"ID": 5, "Nombre": "Les Corts", "Lat": 41.3882, "Lon": 2.1282, "Demanda_kg": 110, "Limite_Hora": 13.0, "Detalle_Hora": "13:00"},
    {"ID": 6, "Nombre": "Sagrada Familia", "Lat": 41.4036, "Lon": 2.1744, "Demanda_kg": 50, "Limite_Hora": 11.5, "Detalle_Hora": "11:30"},
    {"ID": 7, "Nombre": "Raval", "Lat": 41.3792, "Lon": 2.1685, "Demanda_kg": 50, "Limite_Hora": 9.5, "Detalle_Hora": "09:30"},
    {"ID": 8, "Nombre": "Barceloneta", "Lat": 41.3783, "Lon": 2.1901, "Demanda_kg": 80, "Limite_Hora": 12.5, "Detalle_Hora": "12:50"},
    {"ID": 9, "Nombre": "Sarria", "Lat": 41.3992, "Lon": 2.1213, "Demanda_kg": 60, "Limite_Hora": 13.5, "Detalle_Hora": "13:30"},
    {"ID": 10, "Nombre": "Horta", "Lat": 41.4302, "Lon": 2.1612, "Demanda_kg": 30, "Limite_Hora": 14.0, "Detalle_Hora": "14:00"}
]

# Paleta de colores para diferenciar visualmente los distintos viajes
COLORES_RUTAS = ["#1f77b4", "#2ca02c", "#9467bd", "#ff7f0e", "#d62728"]

# 2. ALGORITMO LOGÍSTICO CON RESTRICCIÓN DE CAPACIDAD Y TIEMPOS
def optimizar_flota_ultima_milla(num_clientes, capacidad_max, velocidad_kmh, tiempo_atencion_min, consumo_100km, precio_litro):
    clientes_activos = clientes_v2[:num_clientes]
    no_visitados = clientes_activos.copy()

    viajes = [] # Lista que almacenará cada ruta individual (viajes de la furgoneta)
    total_distancia_flota = 0.0
    total_retrasos = 0

    # El camión inicia su jornada de reparto a las 08:30 AM (8.5 en formato decimal)
    HORA_INICIO_JORNADA = 8.5

    while no_visitados:
        ruta_actual = []
        punto_actual = {"Nombre": "Almacén Central", "Lat": COORDS_DEPOT[0], "Lon": COORDS_DEPOT[1]}
        carga_actual = 0.0
        tiempo_actual = HORA_INICIO_JORNADA
        distancia_ruta = 0.0

        while no_visitados:
            # Buscar el cliente no visitado más cercano que no supere la capacidad restante
            candidato = None
            dist_minima = float('inf')

            for cliente in no_visitados:
                # Comprobamos si cabe en la furgoneta
                if carga_actual + cliente["Demanda_kg"] <= capacidad_max:
                    dist = geodesic((punto_actual["Lat"], punto_actual["Lon"]), (cliente["Lat"], cliente["Lon"])).kilometers
                    if dist < dist_minima:
                        dist_minima = dist
                        candidato = cliente

            # Si no encontramos ningún cliente que quepa, toca regresar al almacén a recargar
            if candidato is None:
                break

            # Viajar al cliente seleccionado
            distancia_ruta += dist_minima
            # Calcular tiempo de viaje: distancia / velocidad (en horas)
            tiempo_viaje = dist_minima / velocidad_kmh
            tiempo_actual += tiempo_viaje

            # Evaluar ventana horaria antes de la descarga
            hora_llegada_texto = f"{int(tiempo_actual):02d}:{int((tiempo_actual%1)*60):02d}"
            retrasado = tiempo_actual > candidato["Limite_Hora"]

            # Realizar la entrega física (suma de tiempo de descarga)
            tiempo_actual += (tiempo_atencion_min / 60.0)
            carga_actual += candidato["Demanda_kg"]

            # Guardamos los datos del paso
            info_parada = candidato.copy()
            info_parada["Hora_Llegada"] = hora_llegada_texto
            info_parada["Retraso"] = retrasado
            info_parada["Distancia_Tramo"] = dist_minima

            ruta_actual.append(info_parada)
            no_visitados.remove(candidato)
            punto_actual = candidato

        # Regresar al Almacén Central al finalizar la carga actual
        dist_regreso = geodesic((punto_actual["Lat"], punto_actual["Lon"]), COORDS_DEPOT).kilometers
        distancia_ruta += dist_regreso

        viajes.append({
            "Ruta": ruta_actual,
            "Carga_Total": carga_actual,
            "Distancia_Total": distancia_ruta
        })
        total_distancia_flota += distancia_ruta

    # 3. CÁLCULO DE MÉTRICAS ECONÓMICAS Y MEDIOAMBIENTALES
    combustible_total = (total_distancia_flota / 100.0) * consumo_100km
    coste_combustible = combustible_total * precio_litro
    emisiones_co2 = combustible_total * 2.64 # 2.64 kg CO2 por litro de diésel

    # --- MOSTRAR MÉTRICAS EN PANTALLA ---
    print("=" * 85)
    print("🚚 PLANIFICADOR INTELIGENTE DE CONTROL DE FLOTAS PRO (v2.0)")
    print("=" * 85)
    print(f"📦 Capacidad Máxima Furgoneta: {capacidad_max} kg | Clientes totales a atender: {len(clientes_activos)}")
    print(f"🛣️  Distancia combinada de flota: {total_distancia_flota:.2f} km")
    print(f"🔄 Número de viajes requeridos: {len(viajes)}")
    print("-" * 85)
    print(f"⛽ Gasoil consumido:  {combustible_total:.2f} L | Coste de ruta: {coste_combustible:.2f} €")
    print(f"🌱 Huella de carbono: {emisiones_co2:.2f} kg CO₂")
    print("=" * 85)

    # 4. RENDERIZADO DEL PLAN DE RUTA DETALLADO
    for idx, viaje in enumerate(viajes):
        color_v = COLORES_RUTAS[idx % len(COLORES_RUTAS)]
        print(f"\n🔄 VIAJE #{idx+1} (Identificado en mapa con color {color_v})")
        print(f"   • Carga transportada: {viaje['Carga_Total']:.1f} kg / {capacidad_max} kg")
        print(f"   • Distancia del viaje: {viaje['Distancia_Total']:.2f} km")
        print(f"   • Secuencia de entrega:")
        print("     [08:30] ➡️ Almacén Central (Salida)")

        for p, parada in enumerate(viaje["Ruta"], 1):
            alerta_tiempo = ""
            if parada["Retraso"]:
                alerta_tiempo = f" 🚨 ¡RETRASO! (Límite: {parada['Detalle_Hora']})"
                total_retrasos += 1
            else:
                alerta_tiempo = " 🟢 A tiempo"

            print(f"     [{parada['Hora_Llegada']}] 👉 Parada {p}: Cliente {parada['Nombre']} | Entregado: {parada['Demanda_kg']} kg |{alerta_tiempo}")
        print("     ➡️ Regreso al Almacén Central")
    print("-" * 85)

    if total_retrasos > 0:
        print(f"⚠️ ATENCIÓN: Se han registrado {total_retrasos} retrasos en la entrega respecto al acuerdo logístico.")
    else:
        print("🏆 ¡Excelente! Todos los envíos se han entregado dentro de su franja horaria pactada.")
    print("=" * 85)

    # 5. GENERAR MAPA INTERACTIVO MULTI-RUTA
    mapa = folium.Map(location=COORDS_DEPOT, zoom_start=13, control_scale=True)

    # Almacén central (Home)
    folium.Marker(
        location=COORDS_DEPOT,
        popup="<b>🏭 Almacén Central (Punto de Carga)</b>",
        tooltip="Depósito Central",
        icon=folium.Icon(color="black", icon="home")
    ).add_to(mapa)

    # Dibujar cada viaje con su color correspondiente
    for idx, viaje in enumerate(viajes):
        color_ruta = COLORES_RUTAS[idx % len(COLORES_RUTAS)]
        puntos_coordenadas = [COORDS_DEPOT]

        for orden, cliente in enumerate(viaje["Ruta"], 1):
            coords_cli = (cliente["Lat"], cliente["Lon"])
            puntos_coordenadas.append(coords_cli)

            # Marcador del cliente
            popup_html = f"""
            <b>👤 Cliente {cliente['Nombre']}</b><br>
            📦 Carga: {cliente['Demanda_kg']} kg<br>
            ⏱️ Llegada: {cliente['Hora_Llegada']}<br>
            🚨 Límite pactado: {cliente['Detalle_Hora']}<br>
            🚲 Viaje: #{idx+1} (Parada {orden})
            """

            folium.Marker(
                location=coords_cli,
                popup=popup_html,
                tooltip=f"Viaje {idx+1} - Parada {orden}",
                icon=folium.Icon(color="red" if cliente["Retraso"] else "blue", icon="shopping-cart", prefix="fa")
            ).add_to(mapa)

        puntos_coordenadas.append(COORDS_DEPOT) # Cerrar de vuelta al depósito

        # Dibujar la línea del viaje
        folium.PolyLine(
            locations=puntos_coordenadas,
            color=color_ruta,
            weight=4,
            opacity=0.8,
            tooltip=f"Trayecto del Viaje #{idx+1}"
        ).add_to(mapa)

    # Mostrar mapa de forma totalmente segura en Google Colab
    mapa_html = mapa._repr_html_()
    display(HTML(f'<div style="width:100%; height:550px; border: 1px solid #ccc; border-radius: 8px; margin-top: 15px;">{mapa_html}</div>'))

# 6. CONSTRUCCIÓN DE LA INTERFAZ GRÁFICA INTERACTIVA
salida_v2 = widgets.Output()

html_t = widgets.HTML("<h4>🚛 Planificador de Reparto con Capacidad y Tiempos</h4>")
in_n_cli = widgets.IntSlider(value=8, min=4, max=10, description='Nº Clientes:')
in_cap = widgets.IntSlider(value=180, min=80, max=400, step=10, description='Capac. (kg):')
in_vel = widgets.FloatSlider(value=25.0, min=15.0, max=50.0, step=2.0, description='Velocidad (km/h):')
in_tiempo_at = widgets.IntSlider(value=15, min=5, max=30, step=5, description='Descarga (min):')
in_cons = widgets.FloatSlider(value=9.0, min=6.0, max=16.0, step=0.5, description='Consumo L/100km:')
in_prec = widgets.FloatText(value=1.58, description='Diésel (€/L):')

boton_calc_v2 = widgets.Button(description='Optimizar Flota Pro', button_style='success', icon='truck-loading')

def disparar_optimizacion_v2(b):
    with salida_v2:
        clear_output(wait=True)
        optimizar_flota_ultima_milla(
            in_n_cli.value, in_cap.value, in_vel.value, in_tiempo_at.value, in_cons.value, in_prec.value
        )

boton_calc_v2.on_click(disparar_optimizacion_v2)

ui_diseno = widgets.VBox([
    html_t,
    widgets.HBox([in_n_cli, in_cap]),
    widgets.HBox([in_vel, in_tiempo_at]),
    widgets.HBox([in_cons, in_prec]),
    widgets.Label(''),
    boton_calc_v2
])

print("🚀 CONTROLADOR DE FLOTAS CAPACITADO (V2.0) INICIADO")
display(ui_diseno, salida_v2)

🚀 CONTROLADOR DE FLOTAS CAPACITADO (V2.0) INICIADO


Output()

In [6]:
#@title PANEL DE CONTROL DE DESPACHOS Y RUTAS INTERNACIONALES
# =====================================================================

# 1. IMPORTACIÓN DE LIBRERÍAS
import sqlite3
import pandas as pd
from datetime import datetime, timedelta
import urllib.request
import json
from IPython.display import HTML, display

# ---------------------------------------------------------------------
# @markdown ### 📥 DATOS DEL EMBARQUE (RELLENA LOS CAMPOS)
# ---------------------------------------------------------------------
Origen = "Puerto de Barcelona ⚓" #@param ["Puerto de Barcelona ⚓", "Aeropuerto de El Prat ✈️", "ZAL Barcelona 📦", "Aduana de La Junquera 🛂"] {allow-input: true}
Destino = "Sabadell" #@param {type:"string"}
Fecha_Recogida = "2026-07-17" #@param {type:"date"}
Hora_Recogida = "11:00" #@param {type:"string"}
Bultos = 12 #@param {type:"slider", min:1, max:100, step:1}
Peso_Mercancia_KG = 500 #@param {type:"number"}

# ---------------------------------------------------------------------
# 2. LOGÍSTICA DE CAPACIDAD (Cálculo del tipo de vehículo aduanero)
# ---------------------------------------------------------------------
def asignar_vehiculo(peso):
    if peso <= 1500:
        return "Furgón Ligero (Courier/Last Mile)", 0.15 # Consumo L/km estimado
    elif peso <= 8000:
        return "Camión Rígido de 2 Ejes (Distribución)", 0.28
    else:
        return "Tráiler Articulado (Tautliner / FTL)", 0.38

vehiculo, consumo = asignar_vehiculo(Peso_Mercancia_KG)

# ---------------------------------------------------------------------
# 3. CÁLCULO ESTIMADO DE DISTANCIAS Y TIEMPOS (SIMULACIÓN DE TRÁNSITO)
# ---------------------------------------------------------------------
# Simulamos una distancia terrestre promedio para la ruta introducida
# (En producción real se conectaría a la API de Google Maps o OpenStreetMap)
distancia_simulada_km = 620 if "Madrid" in Destino else 115
velocidad_promedio_kmh = 80 # Velocidad comercial promedio de camión pesado

tiempo_horas = distancia_simulada_km / velocidad_promedio_kmh
horas_enteras = int(tiempo_horas)
minutos_restantes = int((tiempo_horas - horas_enteras) * 60)

# Cálculo de la hora estimada de llegada (ETA)
datetime_recogida = datetime.strptime(f"{Fecha_Recogida} {Hora_Recogida}", "%Y-%m-%d %H:%M")
eta = datetime_recogida + timedelta(hours=horas_enteras, minutes=minutos_restantes)

# ---------------------------------------------------------------------
# 4. ALMACENAMIENTO DE LA ORDEN DE TRANSPORTE (SQL)
# ---------------------------------------------------------------------
conn = sqlite3.connect('logistica_aduanas.db')
cursor = conn.cursor()

# Creamos la tabla de despachos aduaneros si no existe
cursor.execute('''
CREATE TABLE IF NOT EXISTS Despachos (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    origen TEXT,
    destino TEXT,
    fecha_recogida TEXT,
    hora_recogida TEXT,
    bultos INTEGER,
    peso_kg REAL,
    vehiculo_asignado TEXT,
    eta TEXT
)
''')

# Insertamos la orden configurada por el usuario
cursor.execute('''
INSERT INTO Despachos (origen, destino, fecha_recogida, hora_recogida, bultos, peso_kg, vehiculo_asignado, eta)
VALUES (?, ?, ?, ?, ?, ?, ?, ?)
''', (Origen, Destino, Fecha_Recogida, Hora_Recogida, Bultos, Peso_Mercancia_KG, vehiculo, eta.strftime("%Y-%m-%d %H:%M")))

conn.commit()
conn.close()

# ---------------------------------------------------------------------
# 5. VISUALIZACIÓN DE LA ORDEN DE DESPACHO INTERACTIVA
# ---------------------------------------------------------------------
print("\n" + "="*70)
print("📦 HOJA DE RUTA Y ORDEN DE DESPACHO GENERADA CON ÉXITO")
print("="*70)

html_resumen = f"""
<div style="background-color: #f8f9fa; border-left: 5px solid #1a73e8; padding: 15px; border-radius: 4px; font-family: sans-serif;">
    <h3 style="color: #1a73e8; margin-top: 0;">📋 Detalles Operativos del Envío</h3>
    <table style="width: 100%; border-collapse: collapse;">
        <tr>
            <td style="padding: 6px 0; font-weight: bold; color: #5f6368; width: 35%;">Origen / Aduana de Carga:</td>
            <td style="padding: 6px 0; font-weight: bold; color: #202124;">{Origen}</td>
        </tr>
        <tr>
            <td style="padding: 6px 0; font-weight: bold; color: #5f6368;">Destino Final:</td>
            <td style="padding: 6px 0; color: #202124;">{Destino}</td>
        </tr>
        <tr>
            <td style="padding: 6px 0; font-weight: bold; color: #5f6368;">Fecha y Hora Recogida:</td>
            <td style="padding: 6px 0; color: #202124; font-family: monospace;">{Fecha_Recogida} a las {Hora_Recogida} hs</td>
        </tr>
        <tr>
            <td style="padding: 6px 0; font-weight: bold; color: #5f6368;">Carga Declarada:</td>
            <td style="padding: 6px 0; color: #d93025; font-weight: bold;">{Bultos} bultos | {Peso_Mercancia_KG:,} KG</td>
        </tr>
        <tr style="border-top: 1px solid #dadce0;">
            <td style="padding: 10px 0 6px 0; font-weight: bold; color: #1a73e8;">Vehículo Sugerido:</td>
            <td style="padding: 10px 0 6px 0; color: #1a73e8; font-weight: bold;">{vehiculo}</td>
        </tr>
        <tr>
            <td style="padding: 6px 0; font-weight: bold; color: #1e8e3e;">Hora de Llegada Estimada (ETA):</td>
            <td style="padding: 6px 0; color: #1e8e3e; font-weight: bold; font-family: monospace;">{eta.strftime("%d/%m/%Y a las %H:%M")} hs</td>
        </tr>
    </table>
    <p style="font-size: 0.85em; color: #70757a; margin-top: 15px; border-top: 1px dashed #dadce0; padding-top: 10px;">
        💾 <i>Esta orden ha sido archivada en la tabla SQL de tu base de datos aduanera local (logistica_aduanas.db).</i>
    </p>
</div>
"""
display(HTML(html_resumen))


📦 HOJA DE RUTA Y ORDEN DE DESPACHO GENERADA CON ÉXITO


Origen / Aduana de Carga:,Puerto de Barcelona ⚓
Destino Final:,Sabadell
Fecha y Hora Recogida:,2026-07-17 a las 11:00 hs
Carga Declarada:,12 bultos | 500 KG
Vehículo Sugerido:,Furgón Ligero (Courier/Last Mile)
Hora de Llegada Estimada (ETA):,17/07/2026 a las 12:26 hs
